# B2.7 · Supply chain — SBOM, dependency vulnerabilities, and decompiling the libraries

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *Both directions*

Builds on **[B2.6 · Sandbox replication](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Scan an SBOM against an advisory feed, reconcile it against
what is actually in the built image, and decompile the library that appears in
no manifest.

**Why a security engineer needs it.** A dependency report is a statement about a
manifest, and it is read as a statement about the build. An artefact with no
manifest entry has no identifier, so it has no advisory, so it is counted as
neither vulnerable nor safe — it is not counted. The control it builds is:
reconcile in both directions, then read the constant pool of anything that was
never declared.

This is a **tooling** lesson: it uses a real compiled Java class and recovers its
strings, classes and capabilities without the source.

## 1 · The hook

The dependency report is clean and it is correct. It is also a statement about a manifest, and the jar the booking provider dropped into `lib/` is in no manifest — so it has no identifier, so it has no advisory, so it was counted as neither vulnerable nor safe. It was counted as nothing.

> **At CyberTravels.** The booking provider's integration bundle drops a jar into CyberTravels' image. It is in no manifest, so the weekly dependency report has been silently excluding it, and it carries a hardcoded telemetry endpoint and its own licence key. R5.

## 2 · The framework

```
   WHAT THE DEPENDENCY REPORT COUNTED

   manifest ---> SBOM ---> advisory feed ---> "3 findings / 5 components"
                                                  all of it true

   WHAT IS ACTUALLY IN THE IMAGE

   site-packages/requests        in the SBOM   scanned
   site-packages/sqlalchemy      in the SBOM   scanned
   site-packages/yaml            in the SBOM   scanned  HIGH
   site-packages/jinja2          in the SBOM   scanned
   lib/commons-text-1.9.jar      in the SBOM   scanned  CRITICAL
   lib/vendor-telemetry.jar      ---           NOTHING LOOKED AT IT
                                               no purl -> no advisory
                                                       -> not "safe",
                                                          not counted

   so read the artefact: constant pool, before any bytecode
     CONSTANT_String  -> https://telemetry.vendor-analytics.io/...?k=vnd_live_...
     CONSTANT_String  -> AES/ECB/PKCS5Padding
     CONSTANT_Class   -> java.net.HttpURLConnection, javax.crypto.Cipher

   provable: it CAN reach that endpoint.  not provable: what it sends.
   report the capability. the sentence you cannot evidence takes the
   rest of the report down with it
```

Everything the pipeline has audited so far is code CyberTravels wrote. Most of
what it ships is not.

A dependency scanner answers one question, and answers it well: *are any of the
components I declared known to be vulnerable?* Both halves of that sentence are
limits, and they are the whole subject of this lesson.

**It sees only what was declared.** An SBOM is generated from a manifest. A
vendor's integration bundle that drops a jar into `lib/` is in no manifest, so
it is in no SBOM, so the scanner has no identifier to look up — and the report
comes back clean. That report is accurate. It is a statement about the
manifest, and it is being read as a statement about the build.

**It sees only what is already published.** "No advisory" means nobody has
published one against that identifier yet. For an undeclared artefact there is
no identifier at all, so the absence of an advisory carries no information
whatsoever — and it renders identically on the report to a component that has
genuinely been assessed.

So the procedure has three parts, and only the first is what most teams call
software composition analysis:

1. **Scan what was declared** — match components against an advisory feed, with
   a version comparison that is numeric. `1.9 < 1.10` is false as a string and
   true in every versioning scheme in use, and that one mistake drops findings
   silently.
2. **Reconcile the SBOM against the filesystem** — in both directions. On disk
   and not in the SBOM is undeclared code. In the SBOM and not on disk means
   the SBOM is describing a different build than the one you deployed.
3. **Read the artefacts nobody declared.** No source, no manifest, no
   identifier. This is the part that is treated as exotic and is not: compiled
   code carries its string literals, its class references and its method
   references in a documented table before any bytecode, and recovering them
   needs no vendor cooperation and about forty lines of parser.

That third step is what `jadx`, `procyon`, `Ghidra` and `strings` all start
with. The lesson runs it on a real compiled class and gets a hardcoded endpoint
and a credential-shaped literal out of a library that appears in no manifest.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Two questions, and the one nobody asks

| | a dependency scanner answers | it cannot answer |
|---|---|---|
| **declared** | is this component vulnerable? | is anything here I did not declare? |
| **published** | is there an advisory? | is this artefact dangerous with no advisory? |

The bottom-right cell is where vendored SDKs, agent plugins and MCP server
distributions live. Every one of them arrives as somebody else's compiled code
inside your build, and none of them appears in a report that reads a manifest.

At CyberTravels the booking provider ships an integration bundle. It works, it
was signed off by procurement, and nobody has ever opened it.

## 4 · Where it breaks — the clean report

The scan below finds three real advisories in five declared components, and
every word of it is correct. Then the reconciliation finds a sixth component
that is on disk, in no manifest, and therefore in none of the three numbers
above it.

The failure is not that the scanner is wrong. It is that "3 findings across 5
components" and "3 findings across 5 of 6 components, one unassessed" are the
same report unless somebody counts the filesystem.

## 5 · The stage, as a skill

Three fixtures and one real artefact. The SBOM is CycloneDX, the advisory feed is OSV-shaped, the inventory is what an image scan returns — and `VendorTelemetry.class` is a genuinely compiled Java class, built with `javac` and committed beside the skill. The parser reads its constant pool the way a decompiler does and never sees the source.

The source *is* in the repository, in `evidence/provenance/`, so you can check the recovered strings against it. A vendor does not give you that, which is the reason the step exists.

### The skill — [`skills/appsec/supply-chain-decompile/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/supply-chain-decompile/SKILL.md)

```yaml
name: supply-chain-decompile
description: >-
  Scan an SBOM for known vulnerabilities, reconcile it against what is actually
  on disk, and decompile the dependency that no manifest declares. Use when a
  dependency scan comes back clean, when a third party ships a binary bundle,
  or when asked what a closed-source library in the build actually does.
allowed-tools: Read, Glob, Grep, Bash
```

# What is in the build that the manifest never mentioned?

Dependency scanning answers one question well: *are any of the components I
declared known to be vulnerable?* Both halves of that sentence are limits. It
sees only what was **declared**, and only what is **already published** as an
advisory.

A vendor's integration bundle drops a jar into `lib/`. It is in no manifest,
so it is in no SBOM, so no scanner has an identifier to look up and the report
comes back clean. The clean report is accurate and it is about the manifest,
not about the build.

The step past that is unglamorous and mechanical: **reconcile the SBOM against
the filesystem**, and for anything on disk with no manifest entry, read the
artefact itself. Compiled code carries its strings, its class references and
its method references in a structured table, and recovering them needs no
vendor cooperation and no source.

## When to use this

When any third party ships compiled artefacts into your build — SDKs,
integration bundles, agent plugins, MCP server distributions. Also whenever a
dependency scan is clean on a system that has never had a clean anything, which
usually means the scanner is reading a manifest that stopped describing the
build some time ago.

## Procedure

**1 — Scan what was declared.** Parse the SBOM, match every component against
an advisory feed on `(package, version)` with a real range comparison, and
report each hit with the fixed version. This is the part existing tools do, and
it is worth doing first so the gap that follows is visible against it.

**2 — Reconcile against the filesystem.** List what is actually present in the
built image and diff it against the SBOM's components. Two directions, and both
are findings: **on disk, not in the SBOM** is undeclared code, and **in the
SBOM, not on disk** means the SBOM describes a different build.

**3 — For undeclared artefacts, read the artefact.** No source, no manifest, no
identifier to look up. A compiled class file carries a constant pool: every
string literal, every class it references, every method it calls, in a
documented table before any bytecode. Parse it — the format is stable and the
parser is about forty lines.

**4 — Read the recovered strings and calls as a capability claim.** A URL and
an HTTP client class in the same artefact is an egress capability. A crypto
class beside them tells you what the traffic will look like on the wire. Report
the capability, which is provable from the constant pool, rather than the
intent, which is not.

**5 — Say plainly what the scan could and could not have found.** An
undeclared artefact has no CVE because it has no identifier, not because it is
safe. Those two produce identical output on a dependency report, and only one
of them is a reason to relax.

## Output contract

```json
{
  "sbom": {"components": 0, "vulnerable": 0},
  "advisories": [{"package": "str", "version": "str", "id": "str",
                  "severity": "str", "fixed": "str"}],
  "reconciliation": {"undeclared_on_disk": ["str"], "declared_not_present": ["str"]},
  "decompiled": [{"artefact": "str", "strings": ["str"], "classes": ["str"],
                  "capabilities": ["str"]}],
  "unassessable_by_sbom": 0
}
```

`unassessable_by_sbom` is the number this whole procedure exists to produce. A
report where it is absent has quietly claimed that undeclared code is
vulnerability-free.

## Failure modes

- **Comparing versions as strings.** `1.9 < 1.10` is false lexically and true
  in every version scheme in use. It silently drops the finding.
- **Treating "no advisory" as "no vulnerability".** It means nobody has
  published one against that identifier, and an undeclared artefact has no
  identifier at all.
- **Reconciling in one direction.** Components in the SBOM that are not on disk
  are the same defect seen from the other side: the manifest and the build have
  diverged.
- **Reporting intent from strings.** The constant pool proves an endpoint and a
  client class are present. It does not prove what is sent, and a report that
  says "exfiltrates PII" on that evidence will be dismissed along with the real
  finding next to it.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/supply-chain-decompile/scripts/supply_chain_decompile.py
SCRIPT = "skills/appsec/supply-chain-decompile/scripts/supply_chain_decompile.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 6 · Read the recovered strings again

```
https://telemetry.vendor-analytics.io/v2/ingest?k=vnd_live_8f2c41a09e7b
AES/ECB/PKCS5Padding
java.net.HttpURLConnection   javax.crypto.Cipher
```

The endpoint and the key arrived **concatenated** — the compiler folded them
into a single constant, which is a small gift and a real one: the artefact
carries its own credential in plaintext, and no SBOM, CVE feed or licence scan
would ever have shown you either.

Now the discipline that decides whether this finding survives review. What the
constant pool proves is that this library **can** reach that endpoint and
**can** encrypt before it does. What it sends is not in the constant pool. Write
the finding as a *capability* and it is unarguable; write it as "exfiltrates
traveller PII" and the one sentence you cannot evidence takes the rest of the
report down with it.

`AES/ECB` is worth its own line, and it is the kind of thing this step finds by
accident: ECB mode leaks structure across blocks, so whatever this is
protecting, it is protecting it badly.

## 7 · What to do with it on Monday

Three moves, in cost order:

1. **Count your undeclared artefacts.** Diff the SBOM against the built image.
   The number is almost never zero and almost nobody has it.
2. **Egress-allowlist the workload**, so a capability like the one above cannot
   become a transfer regardless of what the library intended. This is the same
   control as A3.3, arriving from a different direction.
3. **Make the reconciliation a build gate.** An artefact appearing on disk with
   no manifest entry is a build-time event, and it is far cheaper to fail there
   than to find it in a constant pool afterwards.

## What you just proved

Three of five declared components carry published advisories — Text4Shell in commons-text 1.9 as critical, PyYAML 5.3.1 as high, requests 2.31.0 as medium — and commons-text only appears because the version comparison is numeric rather than lexical. Reconciliation then finds one artefact on disk that is in no manifest, which every one of those numbers excluded. Decompiling it recovers, with no source, a hardcoded telemetry endpoint with the licence key folded into the same literal, an AES/ECB cipher spec, and references to `HttpURLConnection` and `javax.crypto.Cipher` — network egress and cryptography as provable capabilities. `unassessable_by_sbom: 1` is the number the report exists to produce.

## Your turn

Generate an SBOM for one service you run, then list what is actually in the built image and diff the two. Every artefact on disk with no manifest entry has been scanned by nothing, and the clean report you have been reading each week was never about it. For the first one you find, pull the strings out of it before you ask the vendor — the conversation goes differently when you open with the endpoint.

---

**Next → [B2.8 · Dynamic exploitation (DAST)](https://spbreed.github.io/cyber-commons/lessons/B2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*